# Track My Subs — אבן דרך 1 (Google Colab)
**מגישות:** עופרי כהן, מיכל זטלאוי

מחברת זו מריצה את מאגר ה-Git מקצה לקצה בסביבת Google Colab נקייה:
1. שכפול המאגר והתקנת תלויות מדויקות (`requirements.txt`).
2. הרצת כלל 33 הבדיקות (`pytest -v`) במצב מדומה (Mock) ללא מפתח API.
3. הדגמת לולאת הסוכן: בברירת מחדל עם `MockDecisionModel` (ללא מפתח), ועם `GeminiDecisionModel` אמיתי כאשר מוגדר `GEMINI_API_KEY` ב-Colab Secrets.
4. הצגת תוצאות ההשוואה האמפירית בין הקשר מלא להקשר מצומצם.


In [ ]:
# שלב 1: שכפול המאגר והתקנת תלויות
REPO_URL = 'https://github.com/YOUR_USERNAME/track-my-subs-agent.git'
import os
if not os.path.exists('track-my-subs-agent'):
    !git clone {REPO_URL}
%cd track-my-subs-agent
!pip install -q -r requirements.txt


In [ ]:
# שלב 2: הרצת כל הבדיקות בפקודה אחת (ללא מפתח API וללא שירות חיצוני)
!pytest -v


In [ ]:
# שלב 3: הדגמת לולאת הסוכן (Mock כברירת מחדל, Gemini אמיתי כשמוגדר GEMINI_API_KEY ב-Colab Secrets)
import os
from src.agent import GeminiDecisionModel, MockDecisionModel, SubscriptionAgent
from src.state import PersistentMemoryStore, SessionStore, SubscriptionRecord
from src.tools import ToolExecutor, TransientError

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMINI_API_KEY')

if api_key:
    print('Mode: Real Gemini Model (GEMINI_API_KEY detected in Colab Secrets / env)')
    decision_model = GeminiDecisionModel(api_key=api_key)
else:
    print('Mode: MockDecisionModel (Default, no API key required)')
    decision_model = MockDecisionModel()

# הרצה 1: תרחיש רגיל — חלופות ל-Netflix
sessions, mem = SessionStore(), PersistentMemoryStore()
mem.upsert_subscription('user_ofri', SubscriptionRecord(
    subscription_id='sub_netflix', service_name='Netflix', amount=69.90, category='streaming'
))
agent = SubscriptionAgent(sessions, mem, ToolExecutor(mem), model=decision_model)
res_normal = agent.run('t_normal', 'user_ofri', 'תראה לי חלופות למנוי NETFLIX שלי', scenario_name='normal_run')
print('\n--- הרצה תקינה ---')
print('סטטוס:', res_normal['status'], '| צעדים:', res_normal['trace']['steps_used'], '| מצב מודל:', res_normal['trace']['model_mode'])
print('תשובה:', res_normal['response'])

# הרצה 2: תרחיש כשל זמני (2 כשלים זמניים -> ניסיונות חוזרים -> התאוששות)
calls = {'n': 0}
def flaky_catalog(tool_name, attempt):
    if tool_name == 'search_alternatives':
        calls['n'] += 1
        if calls['n'] <= 2:
            raise TransientError(f'Mock 503 temporary failure #{calls["n"]}')

mem2 = PersistentMemoryStore()
mem2.upsert_subscription('user_michal', SubscriptionRecord(
    subscription_id='sub_gym', service_name='Holmes Place Gym', amount=250.00, category='fitness'
))
agent2 = SubscriptionAgent(SessionStore(), mem2, ToolExecutor(mem2, fault_injector=flaky_catalog), model=MockDecisionModel())
res_transient = agent2.run('t_transient', 'user_michal', 'הצג לי חלופות למנוי חדר הכושר שלי', retries=2, scenario_name='transient_failure')
print('\n--- הרצה עם כשל זמני והתאוששות ---')
print('סטטוס:', res_transient['status'], '| ניסיונות בכלי החיפוש:', res_transient['trace']['tool_logs'][0]['attempts'])
print('תשובה:', res_transient['response'])


In [ ]:
# שלב 4: הרצת ניסוי ההשוואה בין הקשר מלא להקשר מצומצם
from src.state import run_context_comparison
results = run_context_comparison()
for r in results:
    print(f"{r['strategy_label']} | הודעות: {r['context_messages']} | תווים: {r['chars']} | טוקנים (הערכה): {r['estimated_tokens']} ({r['pct_of_full']}%) | נכונות: {r['answer_correctness']}")
    print(f"   תשובה בפועל: {r['actual_answer']}")
    print(f"   מה אבד: {r['what_is_lost']}\n")
